Inicjalizacja obiektu car do sterowania

In [2]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

print("Zainicjalizowano NvidiaRacecar. Sterowanie gotowe.")

Zainicjalizowano NvidiaRacecar. Sterowanie gotowe.


In [4]:
car.steering_gain = 1.00  
car.steering_offset = 0.0 
car.throttle = -0.25

______________________


Komorka z yolo (wykrywanie klasy person i sterowanie serwo)

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
import ipywidgets
from IPython.display import display

# ===== KONFIG =====
MODEL_PATH = "yolov4_1_3_224_224_static.onnx"
NAMES_PATH = "coco.names"
INPUT_SIZE = 224
CONF_THRESHOLD = 0.3  # Podnieś, jeśli masz za dużo "fałszywych" ramek

# ===== KLASY =====
with open(NAMES_PATH, "r") as f:
    CLASSES = [line.strip() for line in f.readlines()]
    #CLASSES = ["Twoja_Klasa"] # Możesz też wpisać na sztywno, skoro jest jedna

# ===== MODEL (Włączamy TensorRT dla prędkości) =====
providers = [
    ('TensorrtExecutionProvider', {
        'device_id': 0,
        'trt_fp16_enable': True,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': './'
    }),
    'CUDAExecutionProvider'
]

session = ort.InferenceSession(MODEL_PATH, providers=providers)
input_name = session.get_inputs()[0].name

print(f"Model załadowany. Klasa: {CLASSES[0]}")

# ===== KAMERA (Dodano capture_fps) =====
camera = CSICamera(width=224, height=224, capture_device=0, capture_fps=30)
camera.running = True

image_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

try:
    while True:
        frame = camera.value
        if frame is None:
            continue

        # PREPROCESS i INFERENCJA (bez zmian)
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = img.transpose((2, 0, 1)).astype(np.float32) / 255.0
        blob = np.expand_dims(img, axis=0)
        outputs = session.run(None, {input_name: blob})
        
        boxes = np.squeeze(outputs[0])
        scores = np.squeeze(outputs[1])
        target_class_id = 0 
        
        if len(scores.shape) > 1:
            person_scores = scores[:, target_class_id]
        else:
            person_scores = scores

        best_idx = np.argmax(person_scores)
        conf = float(person_scores[best_idx])

        # --- LOGIKA STEROWANIA ---
        # --- LOGIKA STEROWANIA ---
        if conf > CONF_THRESHOLD:
            x1, y1, x2, y2 = boxes[best_idx]
            object_center_x = (x1 + x2) / 2.0
            
            # Parametry omijania
            deadzone = 0.1  # Jeśli obiekt jest w centrum +/- 10%, reagujemy ostro
            
            # Obliczamy bazowe odchylenie
            deviation = object_center_x - 0.5
            
            # LOGIKA OMIJANIA CENTRUM:
            # Jeśli obiekt jest blisko środka (np. między 0.4 a 0.6)
            if abs(deviation) < deadzone:
                # Obiekt na środku! Wymuszamy skręt w lewo (wartość ujemna) 
                # aby "uciec" z kursu kolizyjnego
                avoidance_steering = -0.8 
            else:
                # Obiekt jest już na boku, pogłębiamy skręt w stronę przeciwną
                # Im bliżej krawędzi jest obiekt, tym mniejszy skręt (bo już go omijamy)
                # lub utrzymujemy proporcjonalny:
                avoidance_steering = -deviation * 2.0
            
            # Zastosowanie skrętu
            car.steering = max(min(avoidance_steering, 1.0), -1.0)
            
            # Wizualizacja logiki na ekranie
            color = (0, 0, 255) if abs(deviation) < deadzone else (0, 255, 0)
            ix1, iy1, ix2, iy2 = int(x1*224), int(y1*224), int(x2*224), int(y2*224)
            cv2.rectangle(frame, (ix1, iy1), (ix2, iy2), color, 2)
            cv2.putText(frame, f"CRITICAL" if abs(deviation) < deadzone else "AVOIDING", 
                        (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        else:
            # Jeśli nie widzi człowieka - wyprostuj koła
            car.steering = 0.0

        image_widget.value = bgr8_to_jpeg(frame)

except KeyboardInterrupt:
    car.steering = 0.0
    camera.running = False
    print("Zatrzymano i wyprostowano koła.")


Model załadowany. Klasa: person


Image(value=b'', format='jpeg', height='224', width='224')